# Logistat : préparation des données

**Objectif du projet :** prédire si une commande sera livrée en retard (`Late_delivery_risk` : 1 = retard, 0 = à temps).

**Ce notebook couvre uniquement la préparation :**
1. Charger le jeu d'entraînement
2. Traiter les lignes sans coordonnées
3. Garder les 15 variables retenues
4. Encoder les variables catégorielles
5. Appliquer les mêmes transformations au jeu de test, sans rien y apprendre
6. Enregistrer les jeux préparés

**Règle : on ne touche pas aux données de test.** Toute l'exploration et tous les ajustements (choix, statistiques, encodeur) se font sur l'entraînement seulement. Le test n'est chargé qu'à l'étape 5, pour y appliquer les transformations déjà décidées. `test.csv` n'est jamais modifié.

Les fichiers `train.csv` (85 %) et `test.csv` (15 %) viennent d'une séparation aléatoire stratifiée sur la cible (graine 42). Le choix des 15 variables est expliqué dans `selection_variables.docx`.

## 0. Bibliothèques

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

## 1. Chargement du jeu d'entraînement

Le fichier utilise le point-virgule `;` comme séparateur. Le jeu de test n'est pas chargé ici.

In [ ]:
train = pd.read_csv('train.csv', sep=';')

print('Entraînement :', train.shape)
print('Part de retards :', round(train['Late_delivery_risk'].mean(), 3))

## 2. Traitement des lignes sans coordonnées

Dans le fichier complet, `Latitude` et `Longitude` sont manquantes sur 220 lignes. On les étudie sur l'entraînement seulement.

In [ ]:
na = train.isna().sum()
na[na > 0]

Trois options existaient :
- **Supprimer** les lignes : on perd 220 commandes, peu au regard de 172 765.
- **Imputer** (remplacer par la médiane ou par les coordonnées moyennes de la ville du client).
- **Retirer les colonnes** `Latitude` / `Longitude`.

**Choix retenu : retirer les colonnes.** La position du client est déjà représentée par `Market` et `Order Region`, et `Latitude` / `Longitude` ne font pas partie des 15 variables. On garde ainsi les 220 commandes sans inventer de valeurs. La décision est une règle (retirer deux colonnes), elle s'appliquera telle quelle au test.

On regarde quand même si ces lignes ont un taux de retard différent, pour s'assurer qu'elles ne cachent pas un biais :

In [ ]:
sans_coord = train['Latitude'].isna()
print('Lignes sans coordonnées (train) :', sans_coord.sum())
print('Taux de retard, lignes sans coordonnées :', round(train.loc[sans_coord, 'Late_delivery_risk'].mean(), 3))
print('Taux de retard, autres lignes           :', round(train.loc[~sans_coord, 'Late_delivery_risk'].mean(), 3))

## 3. Sélection des 15 variables

On sépare les variables en deux groupes, car elles ne se traitent pas de la même façon :
- **7 catégorielles** (texte) : à encoder en nombres.
- **8 numériques** : utilisables telles quelles.

Sont exclues : la cible, les colonnes qui révèlent la réponse après coup (`delay_days`, `Delivery Status`, `Days for shipping (real)`, `shipping date`), les identifiants et les colonnes redondantes.

In [ ]:
cible = 'Late_delivery_risk'

variables_cat = ['Type', 'Shipping Mode', 'Customer Segment', 'Category Name',
                 'Department Name', 'Market', 'Order Region']
variables_num = ['Days for shipment (scheduled)', 'Order Item Product Price',
                 'Order Item Quantity', 'Order Item Discount Rate',
                 'Order Item Profit Ratio', 'order_month', 'order_weekday', 'order_hour']
variables = variables_cat + variables_num
assert len(variables) == 15

X_train, y_train = train[variables].copy(), train[cible].copy()

print('X_train :', X_train.shape)
print('Valeurs manquantes restantes :', X_train.isna().sum().sum())

Il ne reste **aucune valeur manquante** : les lignes sans coordonnées sont conservées et complètes sur les 15 variables.

Aperçu des catégories (nombre de valeurs différentes par variable) :

In [ ]:
X_train[variables_cat].nunique().sort_values()

## 4. Encodage des variables catégorielles

**One-hot ou ordinal ?**
- L'encodage **ordinal** (0, 1, 2, …) impose un ordre entre les catégories. Il n'a de sens que si un ordre naturel existe.
- L'encodage **one-hot** crée une colonne 0/1 par catégorie, sans imposer d'ordre.

Nos 7 variables catégorielles sont **nominales** (le paiement, le marché, la région ou la catégorie de produit n'ont pas d'ordre). On utilise donc le **one-hot**. Le seul candidat ordinal serait `Shipping Mode` (Same Day < First < Second < Standard), mais cet ordre est déjà porté par `Days for shipment (scheduled)` (0, 1, 2 et 4 jours), qui reste dans les variables numériques.

**Règles importantes :**
- L'encodeur est **ajusté uniquement sur le jeu d'entraînement** : la liste des catégories vient de train. Il sera ensuite appliqué au test sans être réajusté.
- `handle_unknown='ignore'` : une catégorie absente de l'entraînement mais présente dans le test reçoit des 0 partout, sans erreur.

In [ ]:
encodeur = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encodeur.fit(X_train[variables_cat])

def encoder(X):
    cat = pd.DataFrame(encodeur.transform(X[variables_cat]),
                       columns=encodeur.get_feature_names_out(variables_cat),
                       index=X.index).astype(int)
    return pd.concat([X[variables_num], cat], axis=1)

X_train_enc = encoder(X_train)

print('Avant encodage :', X_train.shape[1], 'variables')
print('Après encodage :', X_train_enc.shape[1], 'colonnes')

Les 15 variables deviennent plusieurs dizaines de colonnes, car chaque catégorie donne une colonne. Aperçu :

In [ ]:
X_train_enc.head()

**Remarque sur la mise à l'échelle :** les variables numériques ont des échelles très différentes (prix jusqu'à 2 000, quantité de 1 à 5). Une standardisation sera utile pour certains modèles (régression logistique, k plus proches voisins), mais pas pour les arbres. On la fera à l'étape de modélisation, selon le modèle choisi.

## 5. Application au jeu de test

Le test est chargé seulement maintenant. On lui applique **exactement** les transformations décidées sur l'entraînement : mêmes 15 variables, même encodeur (`transform` seulement, jamais `fit`). On n'y calcule aucune statistique et on ne modifie pas `test.csv` : le résultat va dans un nouveau fichier.

In [ ]:
test = pd.read_csv('test.csv', sep=';')
X_test, y_test = test[variables].copy(), test[cible].copy()
X_test_enc = encoder(X_test)   # encodeur ajusté sur train uniquement

assert list(X_test_enc.columns) == list(X_train_enc.columns)
print('X_test_enc :', X_test_enc.shape)

## 6. Enregistrement des jeux préparés

On enregistre les variables encodées avec la cible, pour pouvoir passer directement à la modélisation dans un prochain notebook.

In [ ]:
train_prep = X_train_enc.assign(**{cible: y_train})
test_prep = X_test_enc.assign(**{cible: y_test})

train_prep.to_csv('train_prepare.csv', sep=';', index=False)
test_prep.to_csv('test_prepare.csv', sep=';', index=False)
print('train_prepare.csv :', train_prep.shape)
print('test_prepare.csv  :', test_prep.shape)

## Résumé

| Étape | Résultat |
|---|---|
| Chargement | 146 850 lignes en entraînement, 57,3 % de retards |
| Coordonnées manquantes | Colonnes `Latitude` / `Longitude` retirées : aucune ligne perdue, aucune valeur inventée |
| Sélection | 15 variables : 7 catégorielles, 8 numériques |
| Encodage | One-hot sur les 7 catégorielles, ajusté sur l'entraînement seulement |
| Test | Chargé à la fin, transformé avec les règles de l'entraînement, `test.csv` intact |
| Sortie | `train_prepare.csv` et `test_prepare.csv` |

**Étape suivante :** modèle de référence (ci-dessous).

---
# Étape 2 : modèle de base, régression logistique

**But :** obtenir un **point de référence**. Tout modèle plus complexe (forêt aléatoire, boosting…) devra faire mieux que ce score pour valoir la peine.

**Pourquoi la régression logistique ?** Elle est simple, rapide et interprétable : chaque coefficient indique si une variable augmente ou diminue le risque de retard.

**Le test reste intact.** On évalue le modèle par **validation croisée sur le jeu d'entraînement seulement**. Le test servira une seule fois, à la toute fin, pour l'évaluation finale du modèle choisi.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier

## 2.1 Pipeline : mise à l'échelle + modèle

La régression logistique est sensible à l'échelle des variables (prix jusqu'à 2 000, quantité de 1 à 5). On **standardise les 8 variables numériques** (moyenne 0, écart-type 1). Les colonnes one-hot (0/1) restent telles quelles.

La standardisation est placée **dans un pipeline**. Ainsi, à chaque pli de la validation croisée, la moyenne et l'écart-type sont calculés sur la partie d'apprentissage seulement, jamais sur la partie de validation.

In [ ]:
colonnes_onehot = [c for c in X_train_enc.columns if c not in variables_num]

mise_a_echelle = ColumnTransformer([
    ('num', StandardScaler(), variables_num),
    ('cat', 'passthrough', colonnes_onehot),
])

modele_logistique = Pipeline([
    ('echelle', mise_a_echelle),
    ('logit', LogisticRegression(max_iter=2000)),
])

## 2.2 Validation croisée regroupée par commande

On découpe le jeu d'entraînement en **5 plis** : le modèle apprend sur 4 et est évalué sur le 5e, cinq fois de suite.

**Pourquoi regrouper par commande ?** Une commande contient souvent plusieurs articles, qui partagent la même date, le même mode d'expédition et donc le même retard. Si des articles d'une même commande se retrouvaient à la fois en apprentissage et en validation, le score serait trop optimiste. `StratifiedGroupKFold` garde chaque commande (`Order Id`) entière dans un seul pli, tout en conservant environ 57 % de retards dans chaque pli.

In [ ]:
groupes = train['Order Id']
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

## 2.3 Métriques

- **F1** : équilibre entre précision et rappel pour la classe « retard ».
- **Rappel des retards** : part des vrais retards que le modèle détecte. C'est souvent la métrique clé, car un retard non anticipé coûte cher.
- **Précision des retards** : parmi les commandes annoncées en retard, la part qui l'est vraiment.
- **ROC-AUC** : capacité à classer une commande en retard au-dessus d'une commande à temps, indépendamment du seuil de 0,5.
- **Exactitude (accuracy)** : part de bonnes réponses, à comparer au modèle naïf ci-dessous.

In [ ]:
metriques = {'f1': 'f1', 'rappel_retard': 'recall', 'precision_retard': 'precision',
             'roc_auc': 'roc_auc', 'exactitude': 'accuracy'}

def evaluer(modele, nom):
    res = cross_validate(modele, X_train_enc, y_train, groups=groupes, cv=cv, scoring=metriques, n_jobs=-1)
    return pd.Series({m: res[f'test_{m}'].mean() for m in metriques}, name=nom)

## 2.4 Modèle naïf : le plancher

Avant tout, un modèle « naïf » qui prédit toujours la classe majoritaire (retard). Il obtient environ 57 % d'exactitude sans rien apprendre : la régression logistique doit faire nettement mieux.

In [ ]:
naif = evaluer(DummyClassifier(strategy='most_frequent'), 'Naïf (toujours retard)')
logit = evaluer(modele_logistique, 'Régression logistique')

resultats = pd.concat([naif, logit], axis=1).T.round(3)
resultats

## 2.5 Détail des prédictions (validation croisée)

Chaque commande reçoit ici une prédiction faite par un modèle qui ne l'a pas vue pendant l'apprentissage. On peut ainsi construire un rapport complet et une matrice de confusion, toujours sans toucher au test.

In [ ]:
pred_cv = cross_val_predict(modele_logistique, X_train_enc, y_train, groups=groupes, cv=cv, n_jobs=-1)

print(classification_report(y_train, pred_cv, target_names=['À temps', 'En retard'], digits=3))
ConfusionMatrixDisplay.from_predictions(y_train, pred_cv, display_labels=['À temps', 'En retard']);

## 2.6 Coefficients : quelles variables poussent vers le retard ?

On réentraîne le modèle sur tout le jeu d'entraînement pour lire ses coefficients.
- Un coefficient **positif** augmente le risque de retard.
- Un coefficient **négatif** le diminue.

Les variables numériques étant standardisées, leurs coefficients sont comparables entre eux.

In [ ]:
modele_logistique.fit(X_train_enc, y_train)

noms = modele_logistique.named_steps['echelle'].get_feature_names_out()
noms = [n.split('__', 1)[1] for n in noms]
coefs = pd.Series(modele_logistique.named_steps['logit'].coef_[0], index=noms)

print('Variables qui augmentent le plus le risque de retard :')
print(coefs.sort_values(ascending=False).head(10).round(3))
print('\nVariables qui le diminuent le plus :')
print(coefs.sort_values().head(10).round(3))

## Conclusion de l'étape 2

Le tableau de la section 2.4 donne **le score de référence**. Les prochains modèles seront évalués avec **la même validation croisée** (mêmes plis, mêmes métriques), pour une comparaison équitable.

Le jeu de test n'a pas été utilisé : il est réservé à l'évaluation finale du modèle retenu.